# 02_risk_model

Prospective risk model for hypertension new-onset (pooled design with year fixed effects). Includes leakage check via person-level group CV and the adjusted logistic odds ratios (Table 2). Also builds the HTN analysis sample used downstream.

In [1]:
# 02_risk_model.ipynb
# Prospective classifier: t-features -> t+1 new-onset hypertension.
# Pooled over 5 transition intervals with year fixed effects.

import os
import numpy as np
import pandas as pd
import warnings
warnings.filterwarnings("ignore")
import statsmodels.formula.api as smf
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GroupKFold, StratifiedKFold
from sklearn.metrics import roc_auc_score

ROOT = os.path.abspath("..")
DATA = os.path.join(ROOT, "data")
TAB  = os.path.join(ROOT, "results", "tables")

T = pd.read_parquet(os.path.join(DATA, "khp_transitions.parquet"))

# Analysis sample: HTN at-risk (t0 disease-free), complete cases on targets
htn = T[T["HTN"] == 0].copy()
htn["incident"] = (htn["HTN_t1"] == 1).astype(int)
FEATS = ["age", "SEX", "BMI", "smoke_cur", "exer_reg", "walk_days", "WTMG"]
htn = htn.dropna(subset=FEATS + ["incident"])
htn.to_parquet(os.path.join(DATA, "htn_analysis.parquet"))
print("HTN analysis sample:", len(htn),
      "| new-onset:", htn["incident"].sum(),
      f"({htn['incident'].mean()*100:.2f}%)")

HTN analysis sample: 31179 | new-onset: 892 (2.86%)


In [2]:
# Adjusted logistic regression with year fixed effects (Table 2).
htn["female"] = (htn["SEX"] == 2).astype(int)
htn["t0"] = htn["t0"].astype(int)

m = smf.logit(
    "incident ~ BMI + age + female + smoke_cur + exer_reg + walk_days + C(t0)",
    data=htn).fit(disp=0)

OR   = np.exp(m.params)
conf = np.exp(m.conf_int())
tab2 = pd.DataFrame({
    "Variable": ["BMI (+1)", "Age (+1yr)", "Current smoker", "Regular exercise"],
    "OR":     [OR["BMI"], OR["age"], OR["smoke_cur"], OR["exer_reg"]],
    "CI_low": [conf.loc["BMI",0], conf.loc["age",0], conf.loc["smoke_cur",0], conf.loc["exer_reg",0]],
    "CI_high":[conf.loc["BMI",1], conf.loc["age",1], conf.loc["smoke_cur",1], conf.loc["exer_reg",1]],
    "p":      [m.pvalues["BMI"], m.pvalues["age"], m.pvalues["smoke_cur"], m.pvalues["exer_reg"]],
}).round(4)
tab2.to_csv(os.path.join(TAB, "table2_risk_model.csv"), index=False)
print(tab2.to_string(index=False))
print(f"Pseudo R2 = {m.prsquared:.4f}, N = {int(m.nobs)}")

        Variable     OR  CI_low  CI_high      p
        BMI (+1) 1.1088  1.0869   1.1311 0.0000
      Age (+1yr) 1.0522  1.0470   1.0574 0.0000
  Current smoker 1.2938  1.0609   1.5780 0.0110
Regular exercise 0.9075  0.7837   1.0509 0.1947
Pseudo R2 = 0.0681, N = 31179


In [3]:
# Data-leakage check: person-level group CV vs naive CV.
# If AUC barely differs, the model is not memorizing repeat individuals.
FEATS_M = ["BMI", "age", "female", "smoke_cur", "exer_reg", "walk_days"]
X = htn[FEATS_M].astype(float).values
y = htn["incident"].astype(int).values
groups = htn["PIDWON"].values

def cv_auc(splitter, use_groups):
    aucs = []
    it = splitter.split(X, y, groups) if use_groups else splitter.split(X, y)
    for tr, te in it:
        clf = RandomForestClassifier(n_estimators=300, max_depth=6,
                min_samples_leaf=30, class_weight="balanced", random_state=42).fit(X[tr], y[tr])
        aucs.append(roc_auc_score(y[te], clf.predict_proba(X[te])[:, 1]))
    return np.array(aucs)

naive = cv_auc(StratifiedKFold(5, shuffle=True, random_state=42), False)
grp   = cv_auc(GroupKFold(5), True)
print(f"Naive 5-fold AUC        : {naive.mean():.4f} +/- {naive.std():.4f}")
print(f"Person-grouped 5-fold   : {grp.mean():.4f} +/- {grp.std():.4f}")
print(f"Leakage inflation       : {naive.mean()-grp.mean():+.4f}")

Naive 5-fold AUC        : 0.7088 +/- 0.0138
Person-grouped 5-fold   : 0.7148 +/- 0.0161
Leakage inflation       : -0.0060
